In [15]:
from ollama import chat
from ollama import ChatResponse
import secrets
from Crypto.Cipher import AES
from sklearn import cluster
from sklearn.model_selection import train_test_split
from Crypto.Cipher import AES
from Crypto.Hash import SHAKE128
from keras.layers import Input, Dense, Dropout, Flatten, Conv2D, MaxPool2D, Attention, Normalization
import polars as pl
import os
from Crypto.Util.Padding import pad, unpad
import math

In [3]:
'''Passwords & Hashes'''
passwords = [b"Andromeda", b"Neptune", b"Jupiter", b"Saturn"]
algs= ['128', '256'] #be sure to add 512 eventually
hash128 = {password: SHAKE128.new(password).read(128 // 8) for password in passwords}
hash256 = {password: SHAKE128.new(password).read(256 // 8) for password in passwords}
hash512 = {password: SHAKE128.new(password).read(512 // 8) for password in passwords}
print(hash512[b'Andromeda'])
print(hash128)

b"p\x17Y{\xbc\x82\x06\xb7\tr\xd7\xeb)~\xf4\x00%\xf0R\xf73NF\xb3\x86\xd2\x9dJ\x84\x9b(^\x8c\xc3a\x93\xf2\xca\xc6\xafI\xaf\x93\x1f\xe6\x8d\x05)\xb5\xa1\x08\x86X\x02\xc9'\x07\x84\xf3bl\\\x99f"
{b'Andromeda': b'p\x17Y{\xbc\x82\x06\xb7\tr\xd7\xeb)~\xf4\x00', b'Neptune': b'\xe3\xbc\xf4\x89\xd6\x13I4\xe9\x08&\xfd\xe2a[#', b'Jupiter': b'/Z\xd0\xd26\x95\x9b\xe5u\x9bs\xbcub-\x86', b'Saturn': b'L4 \xab\x95\n\xb2\xffM\x14O-\xf1\xe2a\x14'}


In [19]:
'''Encrypt Messages'''
'''need all ciphertexts to be the same size, maybe just set a block limit based on the shortest message length?'''
#should only need to be done once
def encryptMessages(size, pass_dict, max_blocks):
    max_bytes = max_blocks * 512 // 8 #should be a perfect division anyways
    passwords = list(pass_dict.keys())
    for (root, dirs, files) in os.walk("../HumanReadable"):
        for file in files:
            with open(f'../HumanReadable/{file}', 'rb') as plaintextstore:
                plaintext = plaintextstore.read()
                for password in passwords:
                    #print(len(pass_dict[password]))
                    with open(f'../{size}ciphertext/{size}_{password}_{file}', 'xb') as cipherstore:
                        ciphertext = AES.new(pass_dict[password], AES.MODE_ECB).encrypt(pad(plaintext[:max_bytes], 512, 'iso7816'))
                        cipherstore.write(ciphertext)
#not most blocks present but most blocks to use = num blocks in shortest message
max_blocks = 1
for (root, dirs, files) in os.walk("../HumanReadable"):
    for file in files:
        with open(f'../HumanReadable/{file}', 'rb') as plaintextstore:
            plaintext = plaintextstore.read()
            if math.ceil((len(plaintext) * 8) / 512) < max_blocks:
                max_blocks = math.ceil((len(plaintext) * 8) / 512)
encryptMessages('128', hash128, max_blocks)
encryptMessages('256', hash256, max_blocks)

In [22]:
'''Compile Training Data'''
columns = ['alg', 'password', 'plaintext', 'source_file', 'ciphertext']
data = []
for alg in algs:
    for password in passwords:
        for (root, dirs, files) in os.walk("../HumanReadable"):
            for source_file in files:
                with open(f'../HumanReadable/{source_file}', 'rb') as plaintextstore:
                    plaintext = plaintextstore.read()
                    with open(f'../{alg}ciphertext/{alg}_{password}_{source_file}', 'rb') as cipherstore:
                        #If there is an error run encryption again
                        ciphertext = cipherstore.read()
                        print(len(ciphertext))
                        data.append([alg, password, plaintext, source_file, ciphertext])

df = pl.LazyFrame(data, columns, orient='row')
df = df.collect()
df = df.sample(fraction=1, shuffle= True)
train_size = math.floor( .9 * len(df))
test_size = len(df) - train_size
train, test = df.head(train_size), df.tail(test_size)
train_data, test_data = train['ciphertext'], test['ciphertext']
train_targets, test_targets = train.select(pl.exclude('ciphertext')), test.select(pl.exclude('ciphertext'))

512
512
512
512
512
512
512
512
512
512
512
512
512
512
512
512
512
512
512
512
512
512
512
512


In [21]:
train_data

ciphertext
binary
"b""\x1e~\x96_p\x08\xc1\xeaEf\xafQW+\xa9\xf3\xd3g\x1f\xf4o\x14\xf4\x02<sG\xd4\x90\xfa9\x14U\x95\x9d\xf3\x04\x93\xaa\xbc\xc2*\xe2\xa3$\xe5\""0\x92\xa4R\xc6\xc6\xd4X7ua\x1e""…"
"b""\xc9\xcc\xfe\xab\xecJM\xce\xeel\xab\xd7\xfb\x09\x81m8\xd2\x9b\xe2N\xd5N\xdd\x10[\x15\xe9\x97\x0c\x8f\xc6\x0285\x09\x8a\xcb\xa2\x04\x8a\x90\xb9\x06\xda!\x15?h\xb9\xfdL\x8e\xfc\xa5\x8cy\x86\x08l""…"
"b""l\xcc\xc5!\xf1\x8b\x0f\x94\xbd\xa7\xa1\xbe\xe9\xb9\x1d<\x8f\x04\xaaO\x04\xea5\xc9\xe2\xd2}i\xe1\x8c\x10\xc3\x8ci\xb6\xaf\xadSQ$\xb1\x07D\x1dL\x00\xd8\xdf\xa8\xe23\x10]\xb9\xd2\xcf\xb1(e\x17""…"
"b""\x1a\xb9i\xd5\xf8PB\xd4h\xb4Yy\x9e\xc5\xfd\xf0\xb5\xb5\xa6\x03W\xc7\xdai\xb6\x9er\xcf\x1bf[\xbb\xaa\xc8\x95F\xe6\xb1?\x0b\x81`\x05\xf06\x01R1j\xc9+j\x18JY\xc3p\x0f\x8d(""…"
"b""7rI\x12\xcd\xc3\xf6\xe4\x89\xa9\xdf\xd8\x04\xcd3\x1f!dE\x81\x01b\xab\xe1\x9b)kl&O\x0e\x14\x8a\xa4\x9aN\xed\xed\xd7E\x20k9+\x946\xa1\x1cg\xa1\xa5\xda\x85\x10\xf5\xd4\xe1\xadE!""…"
…
"b""\x03#3\xba*\x01\xf8q\xc8\x985\xc0V\xa3\xa6\xeaf\xf2\xd8\x8d$\xe5\xe6\xd3\x15X\xe8\x1e\x0a\x877b\x9do\xad\x8a\xc5\xf5\xe7\xa8\x90\xe3\xe3*E8\xd9\xf7\x09\xc4\x9f\xb7?\xbb=\xf6\xc4<\xe6l""…"
"b""M\xd6fv\xb7\xf4p\xbdNQ\x1cA\xae\xc4q%\x95G\xcdp`sC\xffZ\xeei\xe9\xaf\xc2\x909\x7f6\xc9\xb60\xdc\x16q\xbb_\xb3\x91\xe4T0\xb4\x14\xe56\xd9\xec*\xa0\xf3Xc\x83\xf8""…"
"b""l\xfb\x8d#k\xa8\xaa$P`\x8e\xcd'>d\x0b\xe0!\x12\xfe\xad\x11Vl!K\x1a\x1c_1\xfc\xfd\xf4H\x17\xa6\x0e\xe2IKea\xec1\xe6#\xf3,e\x14\x9d\xe2s\x0d\x8c\x8b\xd4\xa9DU""…"


In [ ]:
'''Autoencoder'''

In [ ]:
'''Supervised Classification'''

In [ ]:
'''Unsupervised Clustering'''

In [ ]:
'''Supervised Confusion'''

In [ ]:
'''Supervised Diffusion'''